# Графики курсовой: импорт Российской Федерации
Данные перенесены из таблиц курсовой. Графики восстановлены по рисункам; исходное оформление может немного отличаться. Запускайте ячейки сверху вниз.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.ticker import FuncFormatter
from IPython.display import display

plt.rcParams.update({"figure.figsize": (11, 6), "font.size": 11, "axes.axisbelow": True})
df = pd.DataFrame([[2006.0, 137807.0, 1060901.0, 27.1355], [2007.0, 199753.0, 1393416.0, 25.5516], [2008.0, 267101.0, 1779109.0, 24.874], [2009.0, 167348.0, 1307927.0, 31.7669], [2010.0, 228912.0, 1633111.0, 30.3765], [2011.0, 305760.0, 2046621.0, 29.3948], [2012.0, 317263.0, 2191486.0, 31.0742], [2013.0, 315298.0, 2288428.0, 31.9063], [2014.0, 287063.0, 2048837.0, 38.6025], [2015.0, 182902.0, 1356703.0, 61.3194], [2016.0, 182448.0, 1280648.0, 66.8335], [2017.0, 227870.0, 1575140.0, 58.2982], [2018.0, 238710.0, 1653005.0, 62.9264], [2019.0, 244573.0, 1695724.0, 64.6184], [2020.0, 232138.0, 1488118.0, 72.323], [2021.0, 293531.0, 1828927.0, 73.6685], [2022.0, 259131.0, 2296410.0, 68.3522], [2023.0, 289148.0, 2034681.0, 85.8116], [2024.0, 282600.0, 2185848.0, 92.6567]], columns=["Год", "Импорт", "ВВП", "Курс"])
df["Год"] = df["Год"].astype(int)
def finish(name):
    plt.grid(alpha=0.22)
    plt.tight_layout()
    plt.savefig(name + '.png', dpi=300, bbox_inches='tight')
    plt.show()
display(df)

## 1. Динамика общего объёма импорта

In [ ]:
plt.figure()
plt.plot(df['Год'], df['Импорт'], 'o-', color='#1679bf', markersize=4)
plt.title('Динамика общего объёма импорта РФ (2006–2024 гг.)')
plt.xlabel('Год'); plt.ylabel('Импорт, млн долл. США')
plt.xticks(df['Год'], rotation=45)
finish('01_import_dynamics')

## 2. Структура импорта в 2024 году
Как на рисунке курсовой, четыре малые группы объединены в «Прочие» (3,8%).

In [ ]:
labels = ['Машины', 'Химия', 'Продовольствие', 'Текстиль', 'Металлы', 'Прочие']
shares = [52.1, 18.9, 13.3, 6.7, 5.2, 3.8]
plt.figure(figsize=(9, 8))
plt.pie(shares, labels=labels, autopct=lambda p: f'{p:.1f}%'.replace('.', ','),
        startangle=90, explode=[0.025,0,0,0,0,0],
        colors=['#acd2e7','#badfcb','#ffd5ad','#f3bec5','#d9c9e7','#e5e3dc'],
        wedgeprops={'edgecolor':'white'})
plt.title('Структура импорта РФ в 2024 году\n(по основным товарным группам)')
plt.tight_layout(); plt.savefig('02_structure.png', dpi=300, bbox_inches='tight'); plt.show()

## 3. Диаграмма рассеяния: ВВП и импорт
Эллипс восстановлен как описательный эллипс рассеяния, не как доверительный интервал регрессии.

In [ ]:
x = df['ВВП'].to_numpy(); y = df['Импорт'].to_numpy()
fig, ax = plt.subplots()
ax.scatter(x, y, color='#1764ff', s=20)
cov = np.cov(x, y)
vals, vecs = np.linalg.eigh(cov)
order = vals.argsort()[::-1]; vals = vals[order]; vecs = vecs[:, order]
angle = np.degrees(np.arctan2(vecs[1,0], vecs[0,0]))
ax.add_patch(Ellipse((x.mean(), y.mean()), 4*np.sqrt(vals[0]), 4*np.sqrt(vals[1]),
                     angle=angle, fill=False, edgecolor='#7395ff', linewidth=1))
ax.set_title('Диаграмма рассеяния: ВВП РФ и импорт РФ (2006–2024)')
ax.set_xlabel('ВВП РФ, млн долл. США'); ax.set_ylabel('Импорт РФ, млн долл. США')
ax.ticklabel_format(style='plain', axis='both')
finish('03_scatter')

## 4. Линейная регрессионная модель

In [ ]:
slope, intercept = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0,1]
plt.figure()
plt.scatter(x, y, color='#add6e5', edgecolor='#82b6ce', label='Фактические данные')
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, intercept + slope*xx, color='#82b6ce', label='Линия регрессии')
plt.text(0.02, 0.97, f'y = {intercept:,.2f} + {slope:.3f}x\nr = {r:.3f}',
         transform=plt.gca().transAxes, va='top', bbox=dict(facecolor='white', edgecolor='lightgray'))
plt.title('Регрессионная модель зависимости импорта РФ от ВВП РФ')
plt.xlabel('ВВП РФ, млн долл. США'); plt.ylabel('Импорт РФ, млн долл. США')
plt.ticklabel_format(style='plain', axis='both'); plt.legend()
finish('04_regression')
print(f'Коэффициент корреляции: {r:.3f}')

## 5. Фактическая и прогнозная динамика
Для точного совпадения подписей с курсовой используются приведённые в ней прогнозы импорта. Это значения из работы, а не новый прогноз.

In [ ]:
future_years = [2025, 2026, 2027]
future_import = [315042, 327170, 339796]
plt.figure()
plt.plot(df['Год'], y, 'o-', color='#86b7cd', markersize=4, label='Фактические данные (2006–2024)')
plt.plot([2024]+future_years, [y[-1]]+future_import, 'o--', color='#e99aaa', markersize=4, label='Прогноз (2025–2027)')
for year, value in zip(future_years, future_import):
    plt.annotate(f'{value:,}'.replace(',', ' '), (year, value), xytext=(0, 8), textcoords='offset points', ha='center')
plt.title('Фактическая и прогнозная динамика импорта РФ (2006–2027 гг.)')
plt.xlabel('Год'); plt.ylabel('Импорт, млн долл. США')
plt.xticks(list(range(2006,2025,2))+future_years, rotation=45); plt.legend()
finish('05_forecast')

## Дополнительно: отраслевая структура по странам
В курсовой это оформлено таблицей. Значения перенесены с изображения; по Беларуси в оригинале указаны данные за 2021 год.

In [ ]:
country_table = pd.DataFrame([
['Китай','Машины и оборудование',63.1,'Чёрные и цветные металлы',6.1,'Текстиль и обувь',5.3],
['Германия','Химическая продукция',44.4,'Машины и оборудование',23.9,'Продовольствие',15.2],
['США','Химическая продукция',36.4,'Машины и оборудование',35.0,'Продовольствие',16.5],
['Турция','Машины и оборудование',36.2,'Продовольствие',21.9,'Химическая продукция',10.5],
['Беларусь (2021)','Машины и оборудование',30.5,'Продовольствие',29.3,'Чёрные и цветные металлы',8.2]
], columns=['Страна','1-е место','Доля 1, %','2-е место','Доля 2, %','3-е место','Доля 3, %'])
display(country_table)